In [16]:
import os
import json
import shutil
from pathlib import Path

def prepare_yolo_dataset(src_root, dest_root):
    """
    Convert COCO dataset to YOLO format while keeping images without annotations (for no-helmet cases).
    """
    splits = ['train', 'valid', 'test']
    dest_paths = {}

    for split in splits:
        dest_paths[split] = {
            'images': os.path.join(dest_root, split, 'images'),
            'labels': os.path.join(dest_root, split, 'labels')
        }
        os.makedirs(dest_paths[split]['images'], exist_ok=True)
        os.makedirs(dest_paths[split]['labels'], exist_ok=True)

    class_mapping = {0: "Helmet", 1: "No Helmet"}  # ✅ Fix class mapping
    class_names = set(class_mapping.values())

    for split in splits:
        src_split_path = os.path.join(src_root, split)
        anno_file = os.path.join(src_split_path, '_annotations.json')

        if not os.path.exists(anno_file):
            print(f"Skipping {split} - annotations.json not found")
            continue

        with open(anno_file) as f:
            data = json.load(f)

        id_to_image = {img['id']: img for img in data['images']}
        
        for ann in data['annotations']:
            if 'bbox' not in ann or not ann['bbox']:
                continue  # Skip invalid annotations

            img_id = ann['image_id']
            img_info = id_to_image.get(img_id)
            if not img_info:
                continue

            src_img_path = os.path.join(src_split_path, img_info['file_name'])
            if not os.path.exists(src_img_path):
                print(f"Image not found: {src_img_path}")
                continue

            img_name = Path(img_info['file_name']).name
            dest_img_path = os.path.join(dest_paths[split]['images'], img_name)
            dest_label_path = os.path.join(dest_paths[split]['labels'], Path(img_name).stem + '.txt')

            shutil.copy2(src_img_path, dest_img_path)

            img_w, img_h = img_info['width'], img_info['height']
            x, y, w, h = ann['bbox']

            x_center = (x + w / 2) / img_w
            y_center = (y + h / 2) / img_h
            w /= img_w
            h /= img_h

            x_center = min(max(x_center, 0), 1)
            y_center = min(max(y_center, 0), 1)
            w = min(max(w, 0), 1)
            h = min(max(h, 0), 1)
            class_id = ann['category_id']
            if class_id > 1:
                print(f"Warning: Invalid class {class_id} detected. Skipping!")
                continue  # Removes class ID 2 warnings

            with open(dest_label_path, 'a') as f:
                f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}\n")

    yaml_content = f"""# YOLO dataset configuration
path: {os.path.abspath(dest_root)}
train: train/images
val: valid/images
test: test/images

# Number of classes
nc: 2

# Class names
names: ['Helmet', 'No Helmet']
"""

    yaml_path = os.path.join(dest_root, 'dataset.yaml')
    with open(yaml_path, 'w') as f:
        f.write(yaml_content)

    print(f"YOLO dataset prepared at: {os.path.abspath(dest_root)}")
    print(f"Dataset YAML file created at: {yaml_path}")


In [17]:
prepare_yolo_dataset('Data_helmet_app2', 'yolo_helm_app2_dataset')

YOLO dataset prepared at: /home/STUDENTS/MSC/DS/22PD/22pd18/DL-package/yolo_helm_app2_dataset
Dataset YAML file created at: yolo_helm_app2_dataset/dataset.yaml
